# ReconFormer Evaluation on fastMRI
EEE 475/575 Project - Accelerated MRI Reconstruction

**Before running:** Set Runtime > Change runtime type > GPU (T4).

Run cells top to bottom. Fill in the paths in the **Config** cell.

## 1. Check GPU

In [ ]:
!nvidia-smi
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

## 2. Clone the ReconFormer repo

In [ ]:
%cd /content
!git clone https://github.com/guopengf/ReconFormer.git
%cd /content/ReconFormer
!ls

## 3. Install dependencies
Only what the eval script actually needs. PyTorch is already in Colab — we don't touch it.

Note: `timm` is pinned to 0.4.12 because the ReconFormer code was written against that API. If a newer version is installed, the Transformer layers may fail to import.

In [ ]:
!pip install -q h5py scikit-image runstats opencv-python tqdm
!pip install -q timm==0.4.12

## 4. Mount Google Drive
Put your fastMRI **val** data and the **pretrained checkpoint** on Drive first, then mount here.

Expected dataset layout (ReconFormer's data loader expects a `PD/val/` structure):
```
<F_path>/PD/val/xxxxx.h5
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 5. Config — EDIT THESE PATHS

In [ ]:
# Path to the fastMRI dataset root. The loader will look inside <F_PATH>/PD/val/
F_PATH = '/content/drive/MyDrive/ReconFormer/fastMRI'   # <-- EDIT

# Path to the pretrained checkpoint file (.pth) downloaded from the repo's OneDrive
CHECKPOINT = '/content/drive/MyDrive/ReconFormer/checkpoints/F_X8_checkpoint.pth'   # <-- EDIT

# Acceleration factor: 8 with center-fraction 0.04, or 4 with center-fraction 0.08
ACCEL = 8
CENTER_FRAC = 0.04

import os
print('F_PATH exists:', os.path.exists(F_PATH))
print('CHECKPOINT exists:', os.path.exists(CHECKPOINT))
val_dir = os.path.join(F_PATH, 'PD', 'val')
print('val dir exists:', os.path.exists(val_dir))
if os.path.exists(val_dir):
    files = [f for f in os.listdir(val_dir) if f.endswith('.h5')]
    print('Number of .h5 files in val:', len(files))

## 6. Inspect one .h5 file (sanity check)
Confirms the data downloaded correctly before running the full eval.

In [ ]:
import h5py, numpy as np
sample = os.path.join(val_dir, sorted(os.listdir(val_dir))[0])
with h5py.File(sample, 'r') as f:
    print('File:', sample)
    print('Keys:', list(f.keys()))
    print('Attributes:', dict(f.attrs))
    for k in f.keys():
        print(f'  {k}: shape={f[k].shape}, dtype={f[k].dtype}')

## 7. Run evaluation
This calls the repo's `main_recon_test.py` directly with the arguments from `run_recon_eval.sh`.

If you hit errors here, copy the full traceback — the most likely culprits are the `timm` API or an old PyTorch call.

In [ ]:
%cd /content/ReconFormer
!python main_recon_test.py \
  --phase test \
  --model ReconFormer \
  --challenge singlecoil \
  --F_path '{F_PATH}' \
  --test_dataset F \
  --sequence PD \
  --accelerations {ACCEL} \
  --center-fractions {CENTER_FRAC} \
  --checkpoint '{CHECKPOINT}' \
  --gpu 0 \
  --verbose

## 8. Notes
- To evaluate at AF=4 as well, set `ACCEL = 4` and `CENTER_FRAC = 0.08` in the Config cell and re-run cells 5 and 7.
- If the eval is slow, you can reduce the number of test scans by moving most `.h5` files out of the `val` folder — 10–20 scans is enough for the project.
- Reconstructed outputs / metrics are saved by the repo's `test_recon_save` function; check the repo's output folder after the run.

## **Now, we are testing for ACCEL = 4 and CENTER_FRAC = 0.08**

In [ ]:
# Path to the fastMRI dataset root. The loader will look inside <F_PATH>/PD/val/
F_PATH = '/content/drive/MyDrive/ReconFormer/fastMRI'   # <-- EDIT

# Path to the pretrained checkpoint file (.pth) downloaded from the repo's OneDrive
CHECKPOINT = '/content/drive/MyDrive/ReconFormer/checkpoints/F_X4_checkpoint.pth'   # <-- EDIT

# Acceleration factor: 8 with center-fraction 0.04, or 4 with center-fraction 0.08
ACCEL = 4
CENTER_FRAC = 0.08

import os
print('F_PATH exists:', os.path.exists(F_PATH))
print('CHECKPOINT exists:', os.path.exists(CHECKPOINT))
val_dir = os.path.join(F_PATH, 'PD', 'val')
print('val dir exists:', os.path.exists(val_dir))
if os.path.exists(val_dir):
    files = [f for f in os.listdir(val_dir) if f.endswith('.h5')]
    print('Number of .h5 files in val:', len(files))

In [ ]:
%cd /content/ReconFormer
!python main_recon_test.py \
  --phase test \
  --model ReconFormer \
  --challenge singlecoil \
  --F_path '{F_PATH}' \
  --test_dataset F \
  --sequence PD \
  --accelerations {ACCEL} \
  --center-fractions {CENTER_FRAC} \
  --checkpoint '{CHECKPOINT}' \
  --gpu 0 \
  --verbose

In [ ]:
import os
val_dir = '/content/drive/MyDrive/ReconFormer/fastMRI/PD/val'
files = [f for f in os.listdir(val_dir) if f.endswith('.h5')]
print('Kaç scan (.h5 dosyası):', len(files))
print('Dosyalar:', files)

import h5py
total = 0
for f in files:
    with h5py.File(os.path.join(val_dir, f), 'r') as hf:
        n = hf['kspace'].shape[0]
        total += n
        print(f'  {f}: {n} slice')
print('Toplam slice:', total)